# Chicago Closure Radar — Step 2: EDA

Explore the data to understand:
- Closure rate distribution by business type
- Review velocity patterns before closure vs. survival
- Sentiment drift timeline
- Rating trajectory for closed vs. open businesses

**Run notebook 01 first to generate the parquet files.**

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

biz   = pd.read_parquet('../data/processed/yelp_chicago_businesses.parquet')
rev   = pd.read_parquet('../data/processed/yelp_chicago_reviews.parquet')
gt    = pd.read_parquet('../data/ground_truth/ground_truth.parquet')

# Merge Yelp with ground-truth labels
biz = biz.merge(gt[['business_id', 'is_closed', 'closure_date']], on='business_id', how='left')
biz['is_closed'] = biz['is_closed'].fillna(False)
print(f'Businesses: {len(biz):,} | Closed: {biz["is_closed"].sum():,} ({100*biz["is_closed"].mean():.1f}%)')

## Closure Rate by Category

In [ ]:
def top_category(cats):
    if not isinstance(cats, str): return 'Other'
    cats = cats.lower()
    if 'book' in cats: return 'Bookstore'
    if 'coffee' in cats or 'cafe' in cats: return 'Café / Coffee'
    if 'bar' in cats: return 'Bar'
    return 'Restaurant'

biz['category'] = biz['categories'].apply(top_category)
closure_rate = biz.groupby('category')['is_closed'].agg(['mean','sum','count']).rename(
    columns={'mean':'closure_rate', 'sum':'n_closed', 'count':'n_total'}
).sort_values('closure_rate', ascending=False)

fig, ax = plt.subplots()
closure_rate['closure_rate'].plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Closure Rate')
ax.set_title('Closure Rate by Business Category — Chicago')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.savefig('../outputs/figures/closure_rate_by_category.png', dpi=150)
plt.show()
closure_rate

## Review Velocity: 12 Months Before Closure vs. Survivors

In [ ]:
closed_ids = set(biz[biz['is_closed']]['business_id'])
open_ids   = set(biz[~biz['is_closed']]['business_id'])

# Align closed businesses to their closure date, open to a random 'event' date
closed_biz = biz[biz['is_closed']].dropna(subset=['closure_date'])

def monthly_velocity(biz_id, anchor_date, rev_df, lookback_months=12):
    start = anchor_date - pd.DateOffset(months=lookback_months)
    sub = rev_df[(rev_df['business_id'] == biz_id) &
                  (rev_df['date'] >= start) &
                  (rev_df['date'] <= anchor_date)].copy()
    sub['month_offset'] = ((sub['date'] - anchor_date) / pd.Timedelta('30D')).apply(int)
    return sub.groupby('month_offset').size()

# Sample up to 200 closed + 200 open businesses for speed
sample_closed = closed_biz.sample(min(200, len(closed_biz)), random_state=42)
sample_open   = biz[~biz['is_closed']].sample(min(200, len(biz[~biz['is_closed']])), random_state=42)

open_anchor = pd.Timestamp('2022-06-01')  # arbitrary anchor for open businesses

def avg_velocity_trend(sample, anchor_col=None, anchor_default=None):
    all_series = []
    for _, row in sample.iterrows():
        anchor = row[anchor_col] if anchor_col and pd.notna(row.get(anchor_col)) else anchor_default
        s = monthly_velocity(row['business_id'], anchor, rev)
        all_series.append(s)
    combined = pd.concat(all_series, axis=1).mean(axis=1)
    return combined.sort_index()

vel_closed = avg_velocity_trend(sample_closed, anchor_col='closure_date')
vel_open   = avg_velocity_trend(sample_open, anchor_default=open_anchor)

fig, ax = plt.subplots()
ax.plot(vel_closed.index, vel_closed.values, label='Closed businesses', color='tomato', lw=2)
ax.plot(vel_open.index, vel_open.values, label='Open businesses', color='steelblue', lw=2, linestyle='--')
ax.axvline(0, color='black', linestyle=':', alpha=0.7, label='Closure event')
ax.set_xlabel('Months before/after closure event')
ax.set_ylabel('Avg reviews/month')
ax.set_title('Review Velocity: Closed vs. Surviving Businesses')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/review_velocity_comparison.png', dpi=150)
plt.show()

## Sentiment Drift Before Closure

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

sample_rev = rev[rev['business_id'].isin(sample_closed['business_id'].tolist()[:50])].copy()
sample_rev['compound'] = sample_rev['text'].fillna('').apply(
    lambda t: sia.polarity_scores(t)['compound']
)

# Join closure date
sample_rev = sample_rev.merge(
    sample_closed[['business_id', 'closure_date']], on='business_id', how='left'
)
sample_rev['days_to_closure'] = (sample_rev['closure_date'] - sample_rev['date']).dt.days
sample_rev = sample_rev[(sample_rev['days_to_closure'] >= 0) &
                         (sample_rev['days_to_closure'] <= 730)]
sample_rev['month_bucket'] = (sample_rev['days_to_closure'] // 30).clip(0, 24)

sent_trend = sample_rev.groupby('month_bucket')['compound'].mean().sort_index(ascending=False)

fig, ax = plt.subplots()
ax.plot(-sent_trend.index, sent_trend.values, color='tomato', lw=2)
ax.axvline(0, color='black', linestyle=':', alpha=0.7)
ax.set_xlabel('Months before closure')
ax.set_ylabel('Avg VADER compound sentiment')
ax.set_title('Sentiment Drift in the 24 Months Before Closure')
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/sentiment_drift_before_closure.png', dpi=150)
plt.show()

## Rating Trajectory: Closed vs. Open

In [ ]:
def rolling_rating(biz_id, anchor_date, rev_df, lookback_months=18):
    start = anchor_date - pd.DateOffset(months=lookback_months)
    sub = rev_df[(rev_df['business_id'] == biz_id) &
                  (rev_df['date'] >= start) &
                  (rev_df['date'] <= anchor_date)].copy()
    sub['month_offset'] = ((sub['date'] - anchor_date) / pd.Timedelta('30D')).apply(int)
    return sub.groupby('month_offset')['stars'].mean()

rat_closed = avg_velocity_trend(sample_closed, anchor_col='closure_date')

all_rat_closed = []
for _, row in sample_closed.head(50).iterrows():
    anchor = row['closure_date'] if pd.notna(row['closure_date']) else pd.Timestamp('2022-06-01')
    s = rolling_rating(row['business_id'], anchor, rev)
    all_rat_closed.append(s)

all_rat_open = []
for _, row in sample_open.head(50).iterrows():
    s = rolling_rating(row['business_id'], open_anchor, rev)
    all_rat_open.append(s)

rat_closed_avg = pd.concat(all_rat_closed, axis=1).mean(axis=1).sort_index()
rat_open_avg   = pd.concat(all_rat_open, axis=1).mean(axis=1).sort_index()

fig, ax = plt.subplots()
ax.plot(rat_closed_avg.index, rat_closed_avg.values, label='Closed', color='tomato', lw=2)
ax.plot(rat_open_avg.index,   rat_open_avg.values,   label='Open',   color='steelblue', lw=2, ls='--')
ax.axvline(0, color='black', linestyle=':', alpha=0.7)
ax.set_xlabel('Months before/after event')
ax.set_ylabel('Avg star rating')
ax.set_title('Star Rating Trajectory: Closed vs. Surviving Businesses')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/rating_trajectory.png', dpi=150)
plt.show()